In [0]:
#Import thư viện và đọc dữ liệu từ CSV

import warnings
warnings.filterwarnings("ignore")

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from mlflow.models import infer_signature
import mlflow

spark = SparkSession.builder.getOrCreate()
mlflow.set_registry_uri("databricks-uc") # Ép ghi nhận vào Unity Catalog

# Đọc file CSV dữ liệu thô
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/smart_gpa/dataset/diem_sinh_vien.csv")

In [0]:
#Tiền xử lý dữ liệu & Gắn nhãn Rớt môn (Điểm F)

# Mã hóa cột chữ 'loai_hoc_phan' sang dạng số (0, 1, 2) để đưa vào mô hình
indexer = StringIndexer(inputCol="loai_hoc_phan", outputCol="loai_hoc_phan_encoded")
df_processed = indexer.fit(df_raw).transform(df_raw)

# Gắn nhãn (Label): Nếu diem_chu_hien_tai == 'F' -> Nhãn = 1.0 (Rớt), ngược lại -> 0.0 (Qua môn)
df_ml_ready = df_processed.withColumn(
    "label",
    when(col("diem_chu_hien_tai") == "F", 1.0).otherwise(0.0)
)

# Gom các cột tính năng vào một Vector duy nhất
feature_cols = ["diem_tich_luy_hien_tai", "diem_trung_binh_thuc_hanh", "tong_so_chi", "loai_hoc_phan_encoded"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

df_final = assembler.transform(df_ml_ready).select("features", "label")
df_final.show(5)

+-----------------+-----+
|         features|label|
+-----------------+-----+
|[8.0,2.9,3.0,0.0]|  0.0|
|[6.3,8.1,3.0,1.0]|  0.0|
|[4.4,3.9,3.0,0.0]|  0.0|
|[9.1,3.0,3.0,1.0]|  0.0|
|[8.5,9.0,3.0,0.0]|  0.0|
+-----------------+-----+
only showing top 5 rows


In [0]:
#Huấn luyện Random Forest và Đăng ký mô hình lên Unity Catalog

# Chia tập Train (80%) và Test (20%)
train_data, test_data = df_final.randomSplit([0.8, 0.2], seed=42)

# Khởi tạo mô hình Random Forest Classifier
rf_fail = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=6, seed=42)

with mlflow.start_run(run_name="Predict_Subject_Failure_F") as run:
    # Huấn luyện mô hình
    model_fail = rf_fail.fit(train_data)
    
    # Dự đoán trên tập kiểm tra
    predictions = model_fail.transform(test_data)
    
    # Đánh giá độ chính xác (Accuracy)
    evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
    accuracy = evaluator.evaluate(predictions)
    print(f"Độ chính xác (Accuracy) mô hình dự báo rớt môn: {round(accuracy * 100, 2)}%")
    
    # Trích xuất Signature cấu trúc API đầu vào/đầu ra chuẩn hóa
    X_train = train_data.select("features")
    y_pred = predictions.select("prediction")
    signature = infer_signature(X_train, y_pred)
    
    # Lưu mô hình an toàn vào Volume tạm thời (Tránh lỗi Serverless JVM)
    mlflow.spark.log_model(
        spark_model=model_fail,
        artifact_path="model",
        signature=signature,
        dfs_tmpdir="/Volumes/workspace/smart_gpa/dataset/mlflow_tmp_fail"
    )
    
    run_id_fail = run.info.run_id

# Đăng ký mô hình vào Unity Catalog Models (Nhớ thay đổi 'workspace' và 'smartgpa_db' nếu khác tên)
full_model_name_fail = "workspace.smartgpa_db.subject_failure_model"
model_details_fail = mlflow.register_model(model_uri=f"runs:/{run_id_fail}/model", name=full_model_name_fail)
print(f"Mô hình dự báo rớt môn đã được đăng ký thành công phiên bản: Version {model_details_fail.version}")

Độ chính xác (Accuracy) mô hình dự báo rớt môn: 100.0%


2026/05/31 14:03:31 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.0.6) contains a local version label (+databricks.connect.18.0.6). MLflow logged a pip requirement for this package as 'pyspark==4.1.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/05/31 14:03:34 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-d493b7aa-89f4-4bb9-aae4-48/tmpw8931npi/model, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 
Registered model 'workspace.smartgpa_db.subject_failure_model' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/24 [00:00<?, ?it/s]

Mô hình dự báo rớt môn đã được đăng ký thành công phiên bản: Version 2


🔗 Created version '2' of model 'workspace.smartgpa_db.subject_failure_model': https://dbc-f659bab4-1de9.cloud.databricks.com/explore/data/models/workspace/smartgpa_db/subject_failure_model/version/2?o=7474656584148137
